In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import os
import json as _json
import matplotlib.pyplot as plt
from IPython.display import display, HTML

from plot_style_utils import (
    apply_plot_theme,
    plot_ratio_distribution,
    plot_annotated_bar,
    display_summary_table,
    PALETTE,
)

# =========================
# USER SETTINGS
# =========================
DATA_DIR = Path("../prof_components_extracted")

MIN_YEAR = 2005
MAX_YEAR = 2024

# Revenue filter
MIN_REVENUE = 1.0

# Flagging thresholds for cost_ratio = (COGS + XSGA_COMPONENTS) / REVT
LOW_THRESHOLD = 0.40
HIGH_THRESHOLD = 1.20

# Extreme thresholds
EXTREME_LOW_THRESHOLD = 0.15
EXTREME_HIGH_THRESHOLD = 2.00

# Plot settings
TOP_N_EXCHANGES = 15
TOP_N_SECTORS = 15

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data folder not found: {DATA_DIR.resolve()}")

apply_plot_theme()

print("Using data from:", DATA_DIR.resolve())


In [ ]:
rows = []
bad_files = []

for fp in sorted(DATA_DIR.glob("*.csv")):
    try:
        df = pd.read_csv(fp)
    except Exception as e:
        bad_files.append((fp.name, f"read_error: {e}"))
        continue

    if "Year" not in df.columns:
        bad_files.append((fp.name, "missing Year"))
        continue

    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df = df.dropna(subset=["Year"]).copy()
    df["Year"] = df["Year"].astype(int)
    df = df[(df["Year"] >= MIN_YEAR) & (df["Year"] <= MAX_YEAR)].copy()

    if df.empty:
        continue

    if "Ticker" not in df.columns:
        df["Ticker"] = fp.stem
    if "firm" not in df.columns:
        df["firm"] = fp.stem

    rows.append(df)

if not rows:
    raise ValueError("No valid CSV files were loaded.")

panel = pd.concat(rows, ignore_index=True)

print(f"Loaded rows: {len(panel):,}")
print(f"Unique tickers: {panel['Ticker'].nunique():,}")

if bad_files:
    print("\nSkipped files:")
    for name, reason in bad_files[:20]:
        print(f"- {name}: {reason}")


In [ ]:
required_cols = ["REVT", "COGS", "XSGA_COMPONENTS"]
missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

for col in required_cols:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

panel["COGS_filled"] = panel["COGS"].fillna(0.0)
panel["XSGA_filled"] = panel["XSGA_COMPONENTS"].fillna(0.0)

panel["REVT_valid"] = panel["REVT"].notna() & (panel["REVT"] > MIN_REVENUE)
panel["TOTAL_COST_PROXY"] = panel["COGS_filled"] + panel["XSGA_filled"]

panel["cost_ratio"] = np.where(
    panel["REVT_valid"],
    panel["TOTAL_COST_PROXY"] / panel["REVT"],
    np.nan
)

# Flag columns
panel["flag_low"]          = panel["cost_ratio"] < LOW_THRESHOLD
panel["flag_high"]         = panel["cost_ratio"] > HIGH_THRESHOLD
panel["flag_extreme_low"]  = panel["cost_ratio"] < EXTREME_LOW_THRESHOLD
panel["flag_extreme_high"] = panel["cost_ratio"] > EXTREME_HIGH_THRESHOLD
panel["flag_any"]          = panel["REVT_valid"] & (panel["flag_low"] | panel["flag_high"])
panel["flag_extreme_any"]  = panel["REVT_valid"] & (panel["flag_extreme_low"] | panel["flag_extreme_high"])

# Extract exchange suffix
def extract_exchange(ticker):
    if pd.isna(ticker):
        return "Unknown"
    ticker = str(ticker)
    if "." in ticker:
        return ticker.split(".")[-1]
    return "Unknown"

panel["Exchange"] = panel["Ticker"].apply(extract_exchange)


In [ ]:
# ── Cost ratio distribution ──────────────────────────────────────────────────
plot_data = panel.loc[panel["cost_ratio"].notna(), "cost_ratio"]

fig, ax = plot_ratio_distribution(
    plot_data,
    low_threshold=LOW_THRESHOLD,
    high_threshold=HIGH_THRESHOLD,
    extreme_low=EXTREME_LOW_THRESHOLD,
    extreme_high=EXTREME_HIGH_THRESHOLD,
)
plt.show()
plt.close(fig)


In [ ]:
# ── Overall summary ──────────────────────────────────────────────────────────
print("=== cost_ratio descriptive statistics ===")
print(panel[["cost_ratio"]].describe().to_string())

print("\nThresholds:")
print(f"  LOW_THRESHOLD          = {LOW_THRESHOLD}")
print(f"  HIGH_THRESHOLD         = {HIGH_THRESHOLD}")
print(f"  EXTREME_LOW_THRESHOLD  = {EXTREME_LOW_THRESHOLD}")
print(f"  EXTREME_HIGH_THRESHOLD = {EXTREME_HIGH_THRESHOLD}")
print(f"  MIN_REVENUE            = {MIN_REVENUE}")

n_total           = len(panel)
n_valid           = int(panel["REVT_valid"].sum())
n_flagged         = int(panel["flag_any"].sum())
n_low             = int(panel["flag_low"].sum())
n_high            = int(panel["flag_high"].sum())
n_extreme         = int(panel["flag_extreme_any"].sum())
n_extreme_low     = int(panel["flag_extreme_low"].sum())
n_extreme_high    = int(panel["flag_extreme_high"].sum())

n_firms_total     = panel["Ticker"].nunique()
n_firms_valid     = panel.loc[panel["REVT_valid"], "Ticker"].nunique()
n_firms_flagged   = panel.loc[panel["flag_any"], "Ticker"].nunique()
n_firms_extreme   = panel.loc[panel["flag_extreme_any"], "Ticker"].nunique()

print("\n=== Flag counts ===")
print(f"  Total rows             : {n_total:,}")
print(f"  Valid ratio rows       : {n_valid:,}")
print(f"  Flagged rows (any)     : {n_flagged:,}  ({n_flagged/n_valid:.1%} of valid)")
print(f"    Low flagged          : {n_low:,}  ({n_low/n_valid:.1%})")
print(f"    High flagged         : {n_high:,}  ({n_high/n_valid:.1%})")
print(f"  Extreme rows (any)     : {n_extreme:,}  ({n_extreme/n_valid:.1%})")
print(f"    Extreme low          : {n_extreme_low:,}")
print(f"    Extreme high         : {n_extreme_high:,}")

print("\n=== Unique companies ===")
print(f"  Total companies        : {n_firms_total:,}")
print(f"  Valid companies        : {n_firms_valid:,}")
print(f"  Flagged companies      : {n_firms_flagged:,}")
print(f"  Extreme companies      : {n_firms_extreme:,}")

# Summary table
flag_summary = pd.DataFrame({
    "Metric": [
        "Total rows", "Valid ratio rows",
        "Flagged rows (any)", "Low flagged", "High flagged",
        "Extreme rows (any)", "Extreme low", "Extreme high",
        "Total companies", "Valid companies",
        "Flagged companies", "Extreme companies",
    ],
    "Count": [
        n_total, n_valid,
        n_flagged, n_low, n_high,
        n_extreme, n_extreme_low, n_extreme_high,
        n_firms_total, n_firms_valid,
        n_firms_flagged, n_firms_extreme,
    ],
})
display_summary_table(flag_summary)


In [ ]:
# ── Flag severity breakdown ───────────────────────────────────────────────────
CATEGORIES     = ["Extreme low", "Moderate low", "Moderate high", "Extreme high"]
CATEGORY_COLORS = {
    "Extreme low":   PALETTE.get("red",    "#d62728"),
    "Moderate low":  PALETTE.get("orange", "#ff7f0e"),
    "Moderate high": PALETTE.get("teal",   "#17becf"),
    "Extreme high":  PALETTE.get("blue",   "#1f77b4"),
}

valid = panel.loc[panel["REVT_valid"] & panel["cost_ratio"].notna()].copy()
valid["flag_category"] = pd.cut(
    valid["cost_ratio"],
    bins=[-np.inf, EXTREME_LOW_THRESHOLD, LOW_THRESHOLD,
          HIGH_THRESHOLD, EXTREME_HIGH_THRESHOLD, np.inf],
    labels=["Extreme low", "Moderate low", "Normal", "Moderate high", "Extreme high"],
    right=False,
)
outside = valid.loc[valid["flag_category"].isin(CATEGORIES)].copy()

obs_counts  = outside["flag_category"].value_counts().reindex(CATEGORIES, fill_value=0)
firm_counts = (
    outside.groupby("flag_category", observed=True)["Ticker"]
    .nunique()
    .reindex(CATEGORIES, fill_value=0)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, counts, title, ylabel in [
    (axes[0], obs_counts,  "Flagged observations by severity",  "Observations"),
    (axes[1], firm_counts, "Unique companies by flag severity",  "Unique companies"),
]:
    bars = ax.bar(
        counts.index, counts.values,
        color=[CATEGORY_COLORS[c] for c in CATEGORIES],
        width=0.5, edgecolor="none",
    )
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(counts.values) * 0.01,
                f"{val:,}", ha="center", va="bottom", fontsize=10)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Flag category")
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(counts.values) * 1.15)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()
plt.close(fig)

severity_table = pd.DataFrame({
    "flag_category":    CATEGORIES,
    "observations":     obs_counts.values,
    "unique_companies": firm_counts.values,
})
display(severity_table)


In [ ]:
# ── Exchange breakdown – all flagged ─────────────────────────────────────────
flagged_all = panel.loc[panel["flag_any"]].copy()

exchange_obs   = flagged_all.groupby("Exchange").size().sort_values(ascending=False).head(TOP_N_EXCHANGES)
exchange_firms = (
    flagged_all.groupby("Exchange")["Ticker"].nunique()
    .sort_values(ascending=False).head(TOP_N_EXCHANGES)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, color in [
    (axes[0], exchange_obs,   "Flagged observations by exchange (all flags)",  PALETTE["blue"]),
    (axes[1], exchange_firms, "Unique flagged firms by exchange (all flags)",   PALETTE["teal"]),
]:
    bars = ax.bar(data.index, data.values, color=color, width=0.5, edgecolor="none")
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(data.values) * 0.01,
                f"{val:,}", ha="center", va="bottom", fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Exchange")
    ax.set_ylabel(ax.get_ylabel() or "Count")
    ax.set_ylim(0, max(data.values) * 1.15)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()
plt.close(fig)


In [ ]:
# ── Exchange breakdown – extreme only ────────────────────────────────────────
flagged_extreme = panel.loc[panel["flag_extreme_any"]].copy()

extr_exch_obs   = flagged_extreme.groupby("Exchange").size().sort_values(ascending=False).head(TOP_N_EXCHANGES)
extr_exch_firms = (
    flagged_extreme.groupby("Exchange")["Ticker"].nunique()
    .sort_values(ascending=False).head(TOP_N_EXCHANGES)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, title, color in [
    (axes[0], extr_exch_obs,   "Extreme observations by exchange",  PALETTE.get("red", "#d62728")),
    (axes[1], extr_exch_firms, "Unique extreme firms by exchange",   PALETTE.get("orange", "#ff7f0e")),
]:
    bars = ax.bar(data.index, data.values, color=color, width=0.5, edgecolor="none")
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(data.values) * 0.01,
                f"{val:,}", ha="center", va="bottom", fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Exchange")
    ax.set_ylabel("Count")
    ax.set_ylim(0, max(data.values) * 1.15)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()
plt.close(fig)

# Exchange summary table – extreme
extr_exch_table = pd.DataFrame({
    "Exchange": extr_exch_obs.index,
    "extreme_observations": extr_exch_obs.values,
    "unique_extreme_firms": extr_exch_firms.reindex(extr_exch_obs.index).fillna(0).astype(int).values,
})
display(extr_exch_table)


In [ ]:
# ── Sector breakdown – all flagged ──────────────────────────────────────────
if "Sector" in panel.columns:
    sector_obs   = flagged_all.groupby("Sector").size().sort_values(ascending=False).head(TOP_N_SECTORS)
    sector_firms = (
        flagged_all.groupby("Sector")["Ticker"].nunique()
        .sort_values(ascending=False).head(TOP_N_SECTORS)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, title, color in [
        (axes[0], sector_obs,   "Flagged observations by sector (all flags)",  PALETTE["orange"]),
        (axes[1], sector_firms, "Unique flagged firms by sector (all flags)",   PALETTE["green"]),
    ]:
        bars = ax.bar(data.index, data.values, color=color, width=0.5, edgecolor="none")
        for bar, val in zip(bars, data.values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(data.values) * 0.01,
                    f"{val:,}", ha="center", va="bottom", fontsize=9)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Sector")
        ax.set_ylabel("Count")
        ax.set_ylim(0, max(data.values) * 1.15)
        ax.tick_params(axis="x", rotation=30)
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()
    plt.close(fig)
else:
    print("Column 'Sector' not found.")


In [ ]:
# ── Sector breakdown – extreme only ─────────────────────────────────────────
if "Sector" in panel.columns:
    extr_sect_obs   = flagged_extreme.groupby("Sector").size().sort_values(ascending=False).head(TOP_N_SECTORS)
    extr_sect_firms = (
        flagged_extreme.groupby("Sector")["Ticker"].nunique()
        .sort_values(ascending=False).head(TOP_N_SECTORS)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, title, color in [
        (axes[0], extr_sect_obs,   "Extreme observations by sector",  PALETTE.get("red", "#d62728")),
        (axes[1], extr_sect_firms, "Unique extreme firms by sector",   PALETTE.get("orange", "#ff7f0e")),
    ]:
        bars = ax.bar(data.index, data.values, color=color, width=0.5, edgecolor="none")
        for bar, val in zip(bars, data.values):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + max(data.values) * 0.01,
                    f"{val:,}", ha="center", va="bottom", fontsize=9)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Sector")
        ax.set_ylabel("Count")
        ax.set_ylim(0, max(data.values) * 1.15)
        ax.tick_params(axis="x", rotation=30)
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # Sector summary table – extreme
    extr_sect_table = pd.DataFrame({
        "Sector": extr_sect_obs.index,
        "extreme_observations": extr_sect_obs.values,
        "unique_extreme_firms": extr_sect_firms.reindex(extr_sect_obs.index).fillna(0).astype(int).values,
    })
    display(extr_sect_table)
else:
    print("Column 'Sector' not found.")


In [ ]:
# ── Build extreme review tables ───────────────────────────────────────────────
# Classify flag type
extreme = panel.loc[panel["flag_extreme_any"]].copy()

extreme["flag_type"] = np.select(
    [
        extreme["flag_extreme_low"] & extreme["flag_extreme_high"],
        extreme["flag_extreme_low"],
        extreme["flag_extreme_high"],
    ],
    ["Both", "Extreme Low", "Extreme High"],
    default="Not flagged",
)

print(f"Extreme rows: {len(extreme):,}")
print(f"Unique extreme firms: {extreme['Ticker'].nunique():,}")

# ── Sort: alphabetically by exchange suffix, then by Ticker ──────────────────
EXCHANGE_ORDER = ["CO", "HE", "IC", "OL", "ST"]   # alphabetical

extreme["exchange_suffix"] = (
    extreme["Ticker"].str.split(".").str[-1].str.upper().fillna("ZZ")
)
order_map = {s: i for i, s in enumerate(EXCHANGE_ORDER)}
extreme["exchange_order"] = (
    extreme["exchange_suffix"].map(order_map).fillna(len(EXCHANGE_ORDER)).astype(int)
)

show_cols = [
    "Ticker", "CompanyName", "Year", "Exchange", "Industry", "Sector",
    "REVT", "COGS", "XSGA_COMPONENTS", "TOTAL_COST_PROXY",
    "cost_ratio", "flag_type",
]
show_cols = [c for c in show_cols if c in extreme.columns]

# ── Split into low / high for separate review ─────────────────────────────────
extreme_sorted = (
    extreme
    .sort_values(["exchange_order", "Ticker", "Year"], ascending=[True, True, True])
    .reset_index(drop=True)
)

extreme_low_table  = extreme_sorted.loc[extreme_sorted["flag_type"] == "Extreme Low",  show_cols].reset_index(drop=True)
extreme_high_table = extreme_sorted.loc[extreme_sorted["flag_type"] == "Extreme High", show_cols].reset_index(drop=True)

print(f"\nExtreme LOW  rows: {len(extreme_low_table):,}  |  unique firms: {extreme_low_table['Ticker'].nunique():,}")
print(f"Extreme HIGH rows: {len(extreme_high_table):,}  |  unique firms: {extreme_high_table['Ticker'].nunique():,}")


In [ ]:
# ── Extreme values statistics ─────────────────────────────────────────────────
print("=== Extreme LOW values (cost_ratio < {}) ===".format(EXTREME_LOW_THRESHOLD))
if len(extreme_low_table):
    print(f"  Rows             : {len(extreme_low_table):,}")
    print(f"  Unique companies : {extreme_low_table['Ticker'].nunique():,}")
    if "Exchange" in extreme_low_table.columns:
        print(f"  Exchanges        : {extreme_low_table['Exchange'].nunique():,}")
        print("  By exchange:")
        for exch, grp in extreme_low_table.groupby("Exchange"):
            print(f"    {exch:8s}  {grp['Ticker'].nunique():3d} firms  |  {len(grp):4d} rows")
    if "Sector" in extreme_low_table.columns:
        print(f"  Sectors          : {extreme_low_table['Sector'].nunique():,}")
        print("  By sector (top 10):")
        for sect, grp in extreme_low_table.groupby("Sector"):
            print(f"    {sect:35s}  {grp['Ticker'].nunique():3d} firms  |  {len(grp):4d} rows")

print()
print("=== Extreme HIGH values (cost_ratio > {}) ===".format(EXTREME_HIGH_THRESHOLD))
if len(extreme_high_table):
    print(f"  Rows             : {len(extreme_high_table):,}")
    print(f"  Unique companies : {extreme_high_table['Ticker'].nunique():,}")
    if "Exchange" in extreme_high_table.columns:
        print(f"  Exchanges        : {extreme_high_table['Exchange'].nunique():,}")
        print("  By exchange:")
        for exch, grp in extreme_high_table.groupby("Exchange"):
            print(f"    {exch:8s}  {grp['Ticker'].nunique():3d} firms  |  {len(grp):4d} rows")
    if "Sector" in extreme_high_table.columns:
        print(f"  Sectors          : {extreme_high_table['Sector'].nunique():,}")
        print("  By sector:")
        for sect, grp in extreme_high_table.groupby("Sector"):
            print(f"    {sect:35s}  {grp['Ticker'].nunique():3d} firms  |  {len(grp):4d} rows")


In [ ]:
# ── Extreme LOW – scrollable review table ────────────────────────────────────
print(f"Extreme LOW  (cost_ratio < {EXTREME_LOW_THRESHOLD}) — sorted alphabetically by exchange, then ticker")
print(f"Rows: {len(extreme_low_table):,}  |  Unique firms: {extreme_low_table['Ticker'].nunique():,}")

display(
    HTML(
        "<div style='max-height:600px; overflow-y:auto'>"
        + extreme_low_table.to_html(index=False, max_rows=None, max_cols=None)
        + "</div>"
    )
)


In [ ]:
# ── Extreme HIGH – scrollable review table ────────────────────────────────────
print(f"Extreme HIGH (cost_ratio > {EXTREME_HIGH_THRESHOLD}) — sorted alphabetically by exchange, then ticker")
print(f"Rows: {len(extreme_high_table):,}  |  Unique firms: {extreme_high_table['Ticker'].nunique():,}")

display(
    HTML(
        "<div style='max-height:600px; overflow-y:auto'>"
        + extreme_high_table.to_html(index=False, max_rows=None, max_cols=None)
        + "</div>"
    )
)
